In [ ]:
%pip show langchain
%pip show langchain-community
%pip show rank-bm25

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
from langchain_core.documents import Document

chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.",
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.",
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

documents = [
    Document(
        page_content=chunk,
        metadata={"source": f"chunk_{i}"}
    )
    for i, chunk in enumerate(chunks)
]

print(f"Created {len(documents)} documents")

In [ ]:
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("Vector retriever ready!")

In [ ]:
test_query = "space exploration company"

print(f"Testing: '{test_query}'")

test_docs = vector_retriever.invoke(test_query)

for doc in test_docs:
    print(f"Found: {doc.page_content}")

In [ ]:
# BM25 Retriever (Keyword Search / Sparse Retrieval)


print("Setting up BM25 Retriever...")

bm25_retriever = BM25Retriever.from_documents(documents)

bm25_retriever.k = 3

print("BM25 retriever ready!")

In [ ]:
test_query = "Cybertruck"

print(f"Testing: '{test_query}'")

test_docs = bm25_retriever.invoke(test_query)

for doc in test_docs:
    print(f"Found: {doc.page_content}")

In [ ]:

# Hybrid Retriever


print("Setting up Hybrid Retriever...")

hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever
    ],
    weights=[0.7, 0.3]
)

print("Setup complete!")

In [ ]:
# Query 1: Mixed semantic + exact terms


test_query = "purchase cost 7.5 billion"

retrieved_chunks = hybrid_retriever.invoke(test_query)

print("Retrieved Documents:\n")

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")

print()
print(
    "Query 1 shows how hybrid retrieval combines "
    "semantic understanding with keyword matching."
)

In [ ]:
# Query 2: Semantic concept + specific product name

test_query = "electric vehicle manufacturing Cybertruck"

retrieved_chunks = hybrid_retriever.invoke(test_query)

print("Retrieved Documents:\n")

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")

print()
print(
    "Query 2 demonstrates combining product-specific "
    "terms with broader concepts."
)

In [ ]:
# Query 3: Semantic + keyword
test_query = "company performance Tesla"

retrieved_chunks = hybrid_retriever.invoke(test_query)

print("Retrieved Documents:\n")

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")

print()
print(
    "Query 3 shows how hybrid retrieval handles "
    "mixed semantic and keyword queries."
)

In [ ]:
# LLM: Local Phi-3 using Ollama

model = ChatOllama(
    model="phi3",
    temperature=0
)

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="phi3",
    temperature=0
)

print("Phi-3 ready!")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Combine the query and retrieved documents

combined_input = f"""
Based on the following documents, please answer this question:

Question:
{test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retrieved_chunks])}

Instructions:
- Answer using only the information from the provided documents.
- Do not make up information.
- If the documents do not contain enough information, say:
  "I don't have enough information to answer that question based on the provided documents."
- Give a clear and concise answer.
"""

messages = [
    SystemMessage(
        content="You are a helpful assistant that answers questions using provided documents."
    ),
    HumanMessage(
        content=combined_input
    )
]

result = model.invoke(messages)

print("\n--- Generated Response ---")
print(result.content)